In [1]:
# Libraries
import numpy as np
import os as os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import netCDF4 as nc
import xarray as xr
import random as rd
import tqdm
import csv
plt.rcParams['figure.dpi'] = 300
import ERA5_process_fxnlib as lib
import pandas as pd

## Check that we have all the ERA5 data corresponding to the GRUAN data (in the proper format)

In [8]:


# This cell retrieves and processes data from ERA5 and GRUAN sources. 
# It creates a list of ERA5 file names, extracts the date and site information from GRUAN file names, and imports latitude and longitude locations of GRUAN sites. ...
# Finally, it picks out the ERA5 files that match the GRUAN dates.

# Make array of .nc filenames (grib)
ERA5_file_names = []
GRUAN_date_sites = []
years = np.linspace(2005,2021,num=17,dtype=int)

# Make list of all ERA5 file names
# Need to loop as glob is not capturing all file names
for i in range(len(years)):
    ERA5_file_names = ERA5_file_names + glob.glob('/home/chinahg/GCresearch/contrailuncertainty/ERA5_processing/ERA5_downloads/ERA5_downloads/'+str(years[i])+'/*.grib', recursive=True)
    ERA5_file_names = ERA5_file_names + glob.glob('/home/chinahg/GCresearch/contrailuncertainty/ERA5_processing/ERA5_downloads/ERA5_downloads/'+str(years[i])+'/*.nc', recursive=True)
ERA5_file_names = list(map(os.path.basename, ERA5_file_names))
ERA5_file_names.sort()

# Stripping GRUAN datetime and site data from file names so we can match with ERA5 data
GRUAN_file_names = [os.path.basename(file) for file in glob.glob('/home/chinahg/GCresearch/GRUAN_sondes/ftp.ncdc.noaa.gov/pub/data/gruan/processing/level2/RS92-GDP/version-002/' + '/**/*.nc', recursive=True)]
GRUAN_num_files = len(GRUAN_file_names)
GRUAN_file_names.sort()

for j in range(GRUAN_num_files):
    GRUAN_date_sites.append(lib.process_GRUAN_filename(GRUAN_file_names[j]))
print(GRUAN_date_sites[2][1])
# Create a list of ERA5 files that match the GRUAN dates
matching_data, matching_files, files2convert, files2download = lib.match_files(GRUAN_date_sites, ERA5_file_names) # Format for matching_data is [ERA5 file name, GRUAN site name, GRUAN datetime object]

if len(files2convert) == 0 and len(files2download) == 0:
    print("All files have been converted and downloaded. Continuing...")
else:
    print("Unconverted files and files to download should be 0 before continuing!")
    # Load the first file in ERA5_file_names as a pandas dataframe


2012-11-28T23:31:00
Matching GRUAN dates with ERA5 files...


100%|██████████| 24196/24196 [00:01<00:00, 12172.67it/s]

Number of unconverted GRIB files:  225
Number of files to download:  468
Number of matching files:  3577
Unconverted files and files to download should be 0 before continuing!


In [3]:
df = xr.open_dataset('/home/chinahg/GCresearch/contrailuncertainty/ERA5_processing/ERA5_downloads/ERA5_downloads/2005/2005_06_08.nc')

# Print the dataframe
print(df)

/home/chinahg/.conda/envs/contrails/lib/python3.9/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.31.0 or higher is recommended. You are running version 2.30.0
  warnings.warn(


<xarray.Dataset>
Dimensions:        (time: 24, isobaricInhPa: 23, latitude: 121, longitude: 1440)
Coordinates:
    number         int64 ...
  * time           (time) datetime64[ns] 2005-06-08 ... 2005-06-08T23:00:00
    step           timedelta64[ns] ...
  * isobaricInhPa  (isobaricInhPa) float64 1e+03 975.0 950.0 ... 225.0 200.0
  * latitude       (latitude) float64 60.0 59.75 59.5 59.25 ... 30.5 30.25 30.0
  * longitude      (longitude) float64 -180.0 -179.8 -179.5 ... 179.5 179.8
    valid_time     (time) datetime64[ns] ...
Data variables:
    r              (time, isobaricInhPa, latitude, longitude) float32 ...
    ciwc           (time, isobaricInhPa, latitude, longitude) float32 ...
    clwc           (time, isobaricInhPa, latitude, longitude) float32 ...
    q              (time, isobaricInhPa, latitude, longitude) float32 ...
    t              (time, isobaricInhPa, latitude, longitude) float32 ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    

In [3]:
# Save data to CSV file
files2download = list(set(files2download))
files2download.sort()
files2download = [str(date).replace("_", "").replace(".nc", "") for date in files2download]

file2download_path = 'files2download.csv'

# Open the file in write mode
with open(file2download_path, 'w', newline='') as csvfile:
    # Create a CSV writer object
    writer = csv.writer(csvfile)
    
    # Write the array to the CSV file
    writer.writerow(files2download)

# Save grib filenames to CSV file
files2convert = list(set(files2convert))
files2convert.sort()

files2convert_path = 'files2convert.csv'

# Open the file in write mode
with open(files2convert_path, 'w', newline='') as csvfile:
    # Create a CSV writer object
    writer = csv.writer(csvfile)
    
    # Write the array to the CSV file
    writer.writerow(files2convert)

In [8]:
# 10km is approx 265 hPa
print("Started!")

cruiseRH = []
MLD_array_pres = []
MLD_top = 0
MLD_bottom = 0
MLD_array_alt = []
press_upper = 250 #hPa
press_lower = 290 #hPa
alt_upper = lib.press2alt(press_upper) # Upper altitude limit [m]
alt_lower = lib.press2alt(press_lower) # Lower altitude limit [m]
num_files = len(matching_files)
redownload = []


for k in tqdm.tqdm(range(num_files)): # Look through all matching files
    current_year = matching_data[k][0][0:4]
    path2file = "/home/chinahg/GCresearch/contrailuncertainty/ERA5_processing/ERA5_downloads/ERA5_downloads/"+str(current_year)+"/"+matching_data[k][0]

    # Read file in the location of interest
    # View data for a single day
    try:
        ds_ERA5 = xr.open_dataset(path2file,engine='netcdf4')
    except:
        ds_ERA5.close()
        # Save the name of the corrupted file to redownload later
        redownload.append(path2file)
        continue
    
    altitudes = lib.press2alt(ds_ERA5.isobaricInhPa.to_numpy())

    # Get site name, coordinates, and time
    current_site_name = matching_data[k][1]
    latitude, longitude = lib.get_coordinates(current_site_name)
    
    if latitude == None or longitude == None:
        continue
    
    time = matching_data[k][2]

    # Assign ERA5 data to arrays
    RH_ERA5 = ds_ERA5.r.sel(time=time, latitude=latitude, longitude=longitude, method='nearest').to_numpy() # [%] Water relative humidity
    T = ds_ERA5.t.sel(time=time, latitude=latitude, longitude=longitude, method='nearest').to_numpy() # [K] Temperature
    q = ds_ERA5.q.sel(time=time, latitude=latitude, longitude=longitude, method='nearest').to_numpy() # [kg/kg] Specific humidity
    p = (ds_ERA5.isobaricInhPa).to_numpy()*100 # [Pa] Pressure
    T0 = 273.15 # [K] Reference temperature
    RH_i = np.zeros(len(T))
    RH_w = np.zeros(len(T))

    # Calculate relative humidity
    for f in range(len(T)):
        P_sat_w = lib.compute_Psat_w(T[f]) # [Pa] Bolton 1980
        P_sat_i =  lib.compute_Psat_i(T[f]) # [Pa] Guide to Meteorological Instruments and Methods of Observation (CIMO Guide) (WMO, 2008)
        RH_w[f] = 0.263*p[f]*q[f]*(np.exp((17.67*(T[f]-T0))/(T[f]-29.65))**(-1)) # [%] Relative humidity wrt water from specific humidity (WMO No.8 Guide to Instruments and Methods of Observation, Vol 1 Measurement of Meteorological Variables, ANNEX 4.B. FORMULAE FOR THE COMPUTATION OF MEASURES OF HUMIDITY)
        
    RH_i = RH_w*P_sat_w/P_sat_i # [%] Relative humidity wrt ice from relative humidity wrt water
        
    RH_len = len(RH_i)

    for i in range(RH_len): # look through all RH datapoints for this date and location
        # Save every RH value
        cruiseRH.append(RH_i[i])

        supersat_bool = lib.check_supersat(RH_i[i], altitudes[i], alt_lower, alt_upper) # Check if supersaturation occurs

        if supersat_bool == True:
            # Save altitude where supersaturated RH starts
            MLD_top = altitudes[i]

            for j in range(i): # Look through list of RH under cruise alt and determine MLD

                if RH_i[i-j] < 100: # MLD ends when RHi < 100%
                    MLD_index = i-j
                    MLD_bottom = altitudes[MLD_index]
                    break
                elif j == i:
                    MLD_bottom = altitudes[0]
                    break
            # Now have an array of RH and MLD upper and lower bounds
            # Take difference of altitudes to get MLD in [m]
            MLD_array_alt.append(MLD_top-MLD_bottom)
        
        else: # If no supersaturation, append 0 to MLD array
            MLD_array_alt.append(0)
    
    ds_ERA5.close()

# MLD_array_alt is appended to for each file, never overwritten
# cruiseRH is appended to for each file, never overwritten

Started!


100%|██████████| 3577/3577 [16:09<00:00,  3.69it/s]


In [9]:
print(redownload)

[]


In [13]:
# Save data to it's correpsonding ERA5 index
# matching_data is a list of lists, each list contains the ERA5 file name, GRUAN site name, and GRUAN datetime object
# We append the cruiseRH and MLD_array_alt to the matching_data list
# matching_data is now a list of lists, each list contains the ERA5 file name, GRUAN site name, GRUAN datetime object, cruiseRH [%], and MLD_array_alt [m]
print("Matching data length: ", len(MLD_array_alt))
for i in range(len(cruiseRH)):
    matching_data[i].append(cruiseRH[i])
    matching_data[i].append(MLD_array_alt[i])

# Sort matching data by the GRUAN site name (to match GRUAN data order)
matching_data.sort(key=lambda x: x[1])

Matching data length:  82271


IndexError: list index out of range

In [ ]:
"""
This code saves data to CSV files and removes existing files if they already exist.

The code performs the following steps:
1. Specify the file paths for MLD, RH, and ERA5 processed data.
2. Check if the MLD, RH, and ERA5 processed files already exist. If they do, remove them.
3. Open the MLD file in write mode and write the MLD array to the CSV file.
4. Open the RH file in write mode and write the cruiseRH array to the CSV file.
5. Open the ERA5 processed file in write mode and write the header and matching data to the CSV file.

Parameters:
- MLD_file_path (str): The file path for the MLD CSV file.
- RH_file_path (str): The file path for the RH CSV file.
- ERA5_processed_path (str): The file path for the ERA5 processed CSV file.
- MLD_array_alt (list): The MLD array to be written to the MLD CSV file.
- cruiseRH (list): The cruiseRH array to be written to the RH CSV file.
- headerList (list): The list of header names for the ERA5 processed CSV file.
- matching_data (list): The matching data to be written to the ERA5 processed CSV file.
- num_files (int): The number of files to iterate over when writing to the ERA5 processed CSV file.

Returns:
None
"""

# Save the data so we don't have to process it again
# Specify the file path
MLD_file_path = 'ERA5_MLD.csv'
RH_file_path = 'ERA5_RH.csv'
ERA5_processed_path = 'ERA5_processed.csv'

# Delete the file if it already exists
if os.path.exists(MLD_file_path):
    os.remove(MLD_file_path)
if os.path.exists(RH_file_path):
    os.remove(RH_file_path)let
if os.path.exists(ERA5_processed_path):
    os.remove(ERA5_processed_path)

# Open the file in write mode
with open(MLD_file_path, 'w', newline='') as csvfile:
    # Create a CSV writer object
    writer = csv.writer(csvfile)
    
    # Write the array to the CSV file
    writer.writerow(MLD_array_alt)

with open(RH_file_path, 'w', newline='') as csvfile:
    # Create a CSV writer object
    writer = csv.writer(csvfile)
    
    # Write the array to the CSV file
    writer.writerow(cruiseRH)

headerList = ['ERA5_file_name', 'GRUAN_site_name', 'GRUAN_datetime', 'cruiseRH', 'MLD'] 
with open(ERA5_processed_path, 'w', newline='') as csvfile:
    # Create a CSV writer object
    writer = csv.DictWriter(csvfile, fieldnames=headerList)
    writer.writeheader()
    # Write the array to the CSV file
    for i in range(num_files): #for i in range(len(matching_data)): FOR TESTING, REMOVE LATER SHOULD BE LEN(MATCHING_DATA)
        writer.writerow({headerList[0]: matching_data[i][0], headerList[1]: matching_data[i][1], headerList[2]: matching_data[i][2], headerList[3]: matching_data[i][3], headerList[4]: matching_data[i][4]})